# Step 4 — Bio_ClinicalBERT Fine-Tuning

Fine-tune `emilyalsentzer/Bio_ClinicalBERT` from HuggingFace Hub.

**Note:** Initial training was done on Google Colab (T4 GPU) because the default
AWS GPU quota was 0. A quota increase was later approved, and the SageMaker
Pipeline Training Job used `ml.g4dn.xlarge`.

Model: https://huggingface.co/emilyalsentzer/Bio_ClinicalBERT

## Option A — SageMaker Training Job (requires ml.g4dn.xlarge quota)

In [ ]:
import boto3
import sagemaker
from sagemaker.huggingface import HuggingFace
from sagemaker.inputs import TrainingInput

BUCKET = "YOUR-S3-BUCKET-NAME"
REGION = "us-east-2"
PREFIX = "symptom-data"

boto_session = boto3.Session(region_name=REGION)
session = sagemaker.Session(boto_session=boto_session, default_bucket=BUCKET)
role = sagemaker.get_execution_role()

estimator = HuggingFace(
    entry_point="train.py",
    source_dir="../scripts",
    role=role,
    instance_type="ml.g4dn.xlarge",   # 1x NVIDIA T4 GPU
    instance_count=1,
    transformers_version="4.26",
    pytorch_version="1.13",
    py_version="py39",
    hyperparameters={
        "epochs":     3,
        "batch-size": 16,
        "max-len":    128,
    },
    sagemaker_session=session,
)

estimator.fit(
    inputs={"train": TrainingInput(s3_data=f"s3://{BUCKET}/{PREFIX}/processed/")},
    wait=True,
    logs=True
)
print("Training complete ✅")

## Option B — Google Colab (no AWS GPU needed)

See `scripts/train.py` — run it in Colab with a T4 GPU, then upload the model to S3.

## Results

In [ ]:
# Expected results after training:
results = {
    "overall_accuracy": 0.7641,
    "macro_f1":         0.73,
    "best_classes":     ["arthritis", "cervical_spondylosis", "fungal_infection"],
    "worst_classes":    ["dengue", "drug_reaction"],
}

print("Training Results:")
print(f"  Overall Accuracy: {results['overall_accuracy']:.1%}")
print(f"  Macro F1:         {results['macro_f1']:.2f}")
print(f"  Best  (F1=1.00):  {results['best_classes']}")
print(f"  Worst (F1=0.00):  {results['worst_classes']}")